# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-04_openness_licences_gpu.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day1/T1-04_openness_licences_gpu.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [ ]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day1"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day1" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)


# T1-04 Openness, licences and GPU questions

**Time:** 13:45–14:20 &nbsp;|&nbsp; **Evidence:** licence scavenger-hunt answers + scenario labels + GPU question card

**Notebook learning outcomes**

- Say what a model licence controls, and find those terms in a real licence file.
- Sort a deployment into prohibited, permitted with obligations, permission required, or unclear (escalate), citing the clause.
- Tell open weights from open source, and say why downloadable does not mean unrestricted.
- Ask why two “24 GB” cards are not the same buy.

This is practice in reading licences, not legal advice. For a real project, your reading goes to counsel and procurement with the clause numbers attached.

**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Ministry link

| Question | Who asked it in the EOIs without using these words |
|---|---|
| May we use it for this job, and what must we do if we share it? | Anyone who would put a model into a service or pass it to another agency |
| Can we see what it was trained on? | Anyone who must **audit** a system (Labour, Health, OPR, Finance). Published data helps audit what went in; it does not explain one particular answer |
| Can we run it here? | **CPO, TTDF, Finance, THA** (N7) |
| Llama vs OLMo | Llama ships under Meta's own community licence plus a use policy; OLMo under Apache 2.0 plus Ai2's guidelines. Both are open-weight; only one licence is OSI-approved |
| Azure already exists | Phi-4 Mini is in the room as the comparison, not the default |

**N4 named:** hosting choice is also an access choice — a public API is a disclosure path.

## Overnight readers

T1-01a put **weights** on disk and T1-03 picked a winner on quality. This notebook adds two gates before anyone builds on that winner: **may we** (the licence and any terms it pulls in) and **can we** (the GPU card). The files you read are in `licences/`, copied from the publishers with checksums, so nobody needs the internet to do this lab.


In [ ]:
# [RUN]
import licence_lab as L

ok = L.verify()   # every licence file matches the checksum recorded when it was downloaded

w = L.t1_03_winner()
if w["winner"] is None:
    print("No T1-03 scorecard on this machine. Pick the model you would build on and carry on.")
elif w.get("scale") == "acceptable":
    print(f"T1-03 winner on acceptable answers: {w['winner']}  (OLMo {w['olmo_score']}, Phi {w['phi_score']})")
else:
    print(f"T1-03 winner on quality: {w['winner']}  (OLMo {w['olmo_score']}/2, Phi {w['phi_score']}/2)")

print()
print("Three gates before a build:")
print("  1. Quality   - T1-03: does it do the job on CivicDesk text?")
print("  2. Permission - today: may we use it this way, and what must we do?")
print("  3. Hardware  - today: can we run it, for how many users, at what cost?")
print("A model that wins gate 1 can still fail gate 2 or 3.")


In [ ]:
# [RUN]
from pathlib import Path
print(Path("data/model_card_excerpts.md").read_text(encoding="utf-8"))


### Teaching note: what a model licence controls

A licence is the publisher's written permission. Read it for seven things:

1. **Use** — may you run it at all, and for which purposes? Is there a separate rule for commercial or government use?
2. **Change** — may you fine-tune it, add an adapter, merge it, quantise it?
3. **Pass it on** — may you give the weights, or your changed version, to someone else? This is different from running a service people use over the web: some conditions only start when you hand over files.
4. **Say so** — what notices, credits, licence copies or naming rules come with passing it on or putting it in a product?
5. **Limits** — is there a cap on users, a list of banned uses, a place or type of organisation that is excluded?
6. **Other documents** — does the licence pull in a use policy or brand rules by link? Those count as part of the deal. A model card may also point to guidelines, data cards or third-party notices; read them too.
7. **What is not promised** — warranty, liability, who defends whom if someone sues.

Two things a licence usually does **not** do: it does not make your use lawful under data-protection law, and it does not make the model accurate. Privacy is a separate obligation that comes from the law and Ministry policy.

**Open weight, open source, closed product**

- **Open weight:** you can download the parameters and run them. The licence may still carry conditions or bans. Llama 3.2 is open weight under Meta's own licence.
- **Open-source licence:** a licence approved by the Open Source Initiative, such as Apache 2.0 or MIT. These let anyone use, change and share for any purpose, with conditions like keeping notices.
- **Open source AI (OSI definition 1.0):** asks for more than a licence on the weights. It wants enough information about the training data for a skilled person to build a substantially equivalent system, the complete training and running code, and the parameters, each under OSI-approved terms. An Apache licence on the weights is one piece of evidence, not the whole check.
- **Closed product:** ChatGPT, Claude, Grok as a service. You get outputs under the provider's terms of service, no weights.

Published training data lets you audit what went in (sources, filtering, licences of the pieces). It does not tell you why the model wrote one particular sentence.


## Scavenger hunt: the Llama 3.2 licence (9 minutes)

You are reading `licences/llama-3.2/LICENSE`, the Llama 3.2 Community License Agreement (release date September 25, 2024), and the use policy it links to. Work in pairs.

For each question, write one of:

- `FOUND: §<section> — "<a few words quoted>"`
- `NOT FOUND` — the licence does not say. That is a correct answer when it is true.
- `NEED LINKED TERMS: <which document>` — the licence points somewhere else for the answer.

Use `L.find("llama32_licence", "word")` to search, or open the file in the Jupyter file browser. The second file is `L.find("llama32_aup", "word")`. Do not ask a chatbot; it is not the licence.

Suggested search words are in brackets. Questions 9 and 10 are for pairs who finish early.


In [ ]:
# [RUN]
L.files()
print()
L.find("llama32_licence", "royalty-free")
L.find("llama32_licence", "training data")   # shows what NOT FOUND looks like


In [ ]:
# [CHANGE] answer each line. Start with FOUND:, NOT FOUND or NEED LINKED TERMS:
hunt = {
    "Q1_use_and_any_commercial_or_government_rule (grant, royalty-free)": "",
    "Q2_may_we_fine_tune (derivative, modifications)": "",
    "Q3_if_we_publish_a_fine_tuned_model_what_must_we_do (distribute, Built with, Notice)": "",
    "Q4_is_there_a_user_limit (monthly active users)": "",
    "Q5_which_uses_are_banned (Acceptable Use)": "",
    "Q5b_now_in_the_AUP_is_military_use_covered (Military)": "",
    "Q6_does_it_say_what_data_trained_the_model (training data, dataset)": "",
    "Q7_does_Meta_promise_the_outputs_are_correct (warranty, AS IS)": "",
    "Q8_does_the_licence_handle_privacy_for_documents_we_feed_it (privacy, laws, personal)": "",
    "Q9_optional_which_courts (Governing Law)": "",
    "Q10_optional_how_can_we_lose_the_licence (terminate)": "",
}
problems = L.check_format({k: v for k, v in hunt.items() if "optional" not in k})
print("\n".join(problems) if problems else "Format OK. Your instructor checks the content.")
import civicdesk_lab as lab
lab.save_log("T1-04_licences", {"hunt": hunt})


## Walkthrough: same job, two licences (instructor-led, 4 minutes)

The two texts:

- **OLMo 2 1B Instruct:** Apache License 2.0 (`licences/apache-2.0/LICENSE-2.0.txt`). Its card also links Ai2's Responsible Use Guidelines, which say they "govern your use" of Ai2 models.
- **Llama 3.2 1B Instruct:** Llama 3.2 Community License Agreement plus the Llama 3.2 Acceptable Use Policy.

The labels you will use:

| Label | Meaning |
|---|---|
| **permitted** | The grant covers it and no condition is triggered |
| **permitted with obligations** | Allowed if you do something: keep a notice, show a credit, follow a policy, name it a certain way |
| **permission required** | The licence says you must ask the publisher, and you have no rights until they agree |
| **prohibited** | A term you agreed to says you may not |
| **unclear - escalate** | The text does not settle it, or a linked document might. Write down the clause and the question, and send it on |

**Scenario S4 (made-up facts):** A defence-force operations cell wants a model on its own servers to summarise intelligence reports for planning military operations. Nothing is published or shared outside.

The next cell prints the clauses. Watch which licence has a term that reaches this job.


In [ ]:
# [RUN]
print("=== Llama 3.2 licence: the clause that pulls in the use policy")
L.find("llama32_licence", "hereby incorporated by reference")
print("=== Llama 3.2 Acceptable Use Policy: item 2 and 2.a")
L.find("llama32_aup", "risk of death or bodily harm")
L.find("llama32_aup", "Military, warfare")
print("=== Apache 2.0: the grant")
L.find("apache", "Grant of Copyright License")
print("=== Ai2 guidelines linked from the OLMo card: any military item?")
L.find("ai2_guidelines", "military")
L.find("ai2_guidelines", "illustrative and not exhaustive")


## Your turn: label four scenarios (5 minutes)

All facts are made up so the clauses can be tested. For each scenario give a label for **each** licence, and the clause you relied on (or `none found`).

**S1. Internal pilot.** A Ministry runs the model on its own server. Staff paste internal policy memos and get summaries. A named officer checks every summary before it is used. No files leave the Ministry.

**S2. Public chatbot.** The Ministry hosts the model on its own server behind a public web page that answers questions about services. Citizens type questions. The weights never leave the server.

**S3. Publishing an adapter.** The Ministry fine-tunes the model for résumé redaction and publishes the result on Hugging Face as **“CivicDesk-Redact-1B”** so other Caribbean governments can download it.

**S5. A very large partner.** The Ministry partners with GlobalChat Inc. GlobalChat and its affiliates had 750 million monthly active users across their products in August 2024. GlobalChat will itself fine-tune and run the model for the Ministry's service.

Useful searches: `L.find("apache", "Redistribution")`, `L.find("llama32_licence", "Built with Llama")`, `L.find("llama32_licence", "700 million")`, `L.find("llama32_aup", "disclose")`.


In [ ]:
# [CHANGE] label: permitted / permitted with obligations / permission required / prohibited / unclear - escalate
scenarios = {
    "S1_OLMo_Apache":  {"class": "", "clause": ""},
    "S1_Llama32":      {"class": "", "clause": ""},
    "S2_OLMo_Apache":  {"class": "", "clause": ""},
    "S2_Llama32":      {"class": "", "clause": ""},
    "S3_OLMo_Apache":  {"class": "", "clause": ""},
    "S3_Llama32":      {"class": "", "clause": ""},
    "S5_OLMo_Apache":  {"class": "", "clause": ""},
    "S5_Llama32":      {"class": "", "clause": ""},
}
problems = L.check_classes(scenarios)
print("\n".join(problems) if problems else "Labels OK. Compare with the room.")
lab.save_log("T1-04_scenarios", {"scenarios": scenarios})


## When it goes to counsel or procurement

Send one short note per model, not a verdict:

1. Exact model name and revision (from the card), and where the licence came from, with the date you read it.
2. The job, in two lines: who uses it, on what data, hosted or shipped, shared with whom.
3. The clauses you think apply, by number, and the label you would give.
4. The open questions, word for word (for example: “Do the Ai2 guidelines bind us on top of Apache 2.0?”, “Is our internal tool ‘made available’ under Llama 3.2 §1.b.i?”).
5. The separate privacy question: what personal data goes in, and which law and Ministry policy cover it.

Recheck the linked documents on the day you decide. Use policies can change after a model is released: the Llama 3.2 policy you read today has different wording from the version published in September 2024 (both are in `licences/llama-3.2/`).


## GPU question card

**4 minutes in the room:** answer questions 1, 2 and 6 for this machine. Questions 3 to 5 go home with you tonight.

Ministries hear “we need a GPU” and buy one box for everything. Split **burst** (a short fine-tune, Day 3 / Tier 2) from **standing inference** (this chat API). VRAM is necessary and not sufficient.

Fill `cards/gpu_questions.md` — do not memorise SKUs:

1. Burst or standing, and how many users?
2. How much VRAM, at what precision?
3. Architecture / compute capability — will vLLM actually run?
4. Compute-bound or memory-bandwidth-bound?
5. Consumer vs datacentre (ECC, multi-user, driver, licence)?
6. Can it idle-stop?

`# [OPTIONAL]` look at `lab.gpu_note()` again and write which of the six questions you can already answer for *this* machine.


In [ ]:
# [OPTIONAL]
import civicdesk_lab as lab
print(lab.gpu_note())


`# [OPTIONAL]` After class: `tier1/deep/T1-04d_licences_deep.ipynb` compares Meta Llama 3 with Llama 3.2, shows how the use policy changed, looks at Phi's MIT licence and third-party notice, and reads the licences of OLMo's fine-tuning data. Nothing in it is needed for Day 2.


## Write-up

1. In one line: OLMo, Phi, ChatGPT — what can you *see*, and what are you *allowed* to do with it?
2. Your T1-03 winner: which of its licence terms would matter most for a Ministry service, and which clause says so?
3. Why might TTDF refuse a pretty public API even if it scored higher in T1-03?
4. Which GPU question from the card could you not answer today, and who in your Ministry could?
